# How to Compute the Conjunctive Normal Form

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [ ]:
import { parse, Formula, Variable } from './PropositionalLogicParser';

Furthermore, we use the library `recursive-set`.  This library implements sets in a way such that the elements of sets are compared structurally and not by reference.

In [ ]:
import { RecursiveSet as Set, Tuple, Value, flatMap } from 'recursive-set';

In [ ]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [ ]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

In [ ]:
tpl('a', 'b')

In [ ]:
type Literal  = Variable 
              | Tuple<['¬', Variable]>;
type Clause   = Set<Literal>;
type CNF      = Set<Clause>;

In [ ]:
let f = parse("p ↔ q");
f

The function `eliminateBiconditional(f)` takes a formula `f` from propositional logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (g \leftrightarrow h) \;\Leftrightarrow\; (g \rightarrow h) \wedge (h \rightarrow g) $$

In [ ]:
type Formula1 = Variable 
              | ['⊤' | '⊥']
              | ['¬', Formula1]
              | ['→' | '∧' | '∨', Formula1, Formula1];

In [ ]:
function eliminateBiconditional(f: Formula): Formula1 {
    if (typeof f == 'string') { return f; }
    switch (f[0]) {
        case '⊤':
        case '⊥': return f;        
        case '¬': const [_, g] = f; return ['¬', eliminateBiconditional(g)];
        default: 
            const [op, h, k] = f;
            const [hs, ks] = [h, k].map(eliminateBiconditional);
            switch (op) {   
                case '↔': return ['∧', ['→', hs, ks], ['→', ks, hs]]; 
                default: return [op, hs, ks]; 
}}}

In [ ]:
let f1 = eliminateBiconditional(f);
f1

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from propositional logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (g \rightarrow h) \;\Leftrightarrow\; (\neg g \vee h) $$

In [ ]:
type Formula2 = Variable 
              | ['⊤' | '⊥'] 
              | ['¬', Formula2]
              | ['∧' | '∨', Formula2, Formula2];

In [ ]:
function eliminateConditional(f: Formula1): Formula2 {
    if (typeof f == 'string') { return f; }
    switch (f[0]) {
        case '⊤':
        case '⊥': return f;
        case '¬': const [_, g] = f; return ['¬', eliminateConditional(g)]; 
        default: 
            const [op, h, k] = f;
            const [hs, ks] = [h, k].map(eliminateConditional);
            switch (op) {
                case '→': return ['∨', ['¬', hs], ks];
                default:  return [op, hs, ks];                              
}}}

In [ ]:
let f2 = eliminateConditional(f1);
f2

In [ ]:
type NNF = Variable 
         | ['⊤' | '⊥'] 
         | ['¬', Variable] 
         | ['∧' | '∨', NNF, NNF];

The function $\texttt{nnf}(f)$ computes the *negation normal form* of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

As the functions `nnf` and `neg` are mutually recursive, we need to define together them in a single cell.

In [ ]:
function nnf(f: Formula2): NNF {
    if (typeof f == 'string') { return f; }
    switch (f[0]) {
        case '⊤':
        case '⊥': return f;
        case '¬': const [_, g] = f; return neg(g);    
        default:  const [op, h, k] = f; return [op, nnf(h), nnf(k)]; 
}}

function neg(f: Formula2): NNF {
    if (typeof f == 'string') { return ['¬', f]; }
    switch (f[0]) {
        case '⊤': return ['⊥'];
        case '⊥': return ['⊤'];
        case '¬': const [_, g] = f; return nnf(g);    
        default:  const [op, h, k] = f;
                  return [op == '∧' ? '∨' : '∧', neg(h), neg(k)]; 
}}

The function $\texttt{cnf}(f)$ takes a formula $f$ that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables and returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

In [ ]:
function cnf(f: NNF): CNF {
    if (typeof f == 'string') { return set(set(f)); }
    switch (f[0]) {
        case '⊤': return set();     
        case '⊥': return set(set());
        case '¬': const [_, p] = f; return set(set(tpl('¬', p))); 
        default: 
            const [op, h, k] = f;
            const [hs, ks] = [h, k].map(cnf); 
            switch (op) { 
                case '∧': return hs.union(ks);
                case '∨': return hs.cartesianProduct(ks).map(([c, d]) => c.union(d));
}}}

The function `getComplement` computes the complement $\overline{l}$ of a literal $l$. 
If $p$ is a propositional variable, the complement is defined as follows:
 * $\overline{p} = \neg p$,
 * $\overline{\neg p} = p$.  

In [ ]:
function getComplement(l: Literal): Literal {
    if (typeof l == 'string') { return tpl('¬', l); }
    return l.get(1);
}

The function $\texttt{isTrivial}(C)$ checks whether the clause $C$ is *trivial*.

In [ ]:
function isTrivial(clause: Clause): boolean {
    return clause.some(l => clause.has(getComplement(l)));
}

The function $\texttt{simplify}(Cs)$ takes a set of clauses and removes all trivial clauses from $Cs$.

In [ ]:
function simplify(clauses: CNF): CNF {
    return clauses.filter(c => !isTrivial(c));
}

We know that the following equivalence holds:
$$ C \wedge (C \vee D) \Leftrightarrow C $$ 
For this reason, if we have a set $S$ of clauses containing to clauses $C_1$ and $C_2$ such that
$$ C_1 \subset C_2,$$
then we can drop the clause $C_2$ from $S$ since it is *subsumed* by $C_1$.

The function `findSubsumed` takes a set of clauses $S$ and a clause $C_1$ and returns the set of all clauses $C_2$ from $S$ that are subsumed by $C_1$.  These clauses will later be removed from $S$.

In [ ]:
function findSubsumed(S: CNF, C1: Clause): CNF {
    return S.filter(C2 => !C2.equals(C1) && C1.isSubset(C2));
}

Given a set $S$ of clauses, the function `subsume` removes those clauses from $S$ that are subsumed by another clause in $S$.

In [ ]:
function subsume(S: CNF): CNF {
    return S.difference(flatMap(S, cl => findSubsumed(S, cl)));
}

The function `pipe` takes a value $v$ and a list $[f_1, f_2, ..., f_n]$ of unary functions.
It returns the value $f_n(\cdots f_2(f_1(v))\cdots)$.

In [ ]:
function pipe(v: any, ...fns: Function[]): any {
    return fns.reduce((acc, fn) => fn(acc), v);
}

The function $\texttt{normalize}$ takes a propositional formula $f$ and transforms $f$ into *conjunctive normal form*.  
Furthermore, both *trivial* clauses and clauses that are *subsumed* by simpler clauses are removed.

In [ ]:
function normalize(f: Formula): CNF {
    return pipe(f,
                eliminateBiconditional, // Formula  -> Formula1
                eliminateConditional,   // Formula1 -> Formula2
                nnf,                    // Formula2 -> NNF
                cnf,                    // NNF      -> CNF
                simplify,               // remove trivial clauses
                subsume);               // remove subsumed clauses
}

The function `prettify` takes a set of clauses and converts them into a string.

In [ ]:
function prettify(M: CNF): string {
    const clauseStrings = [...M.map(clause => {
        const literalStrings = [...clause.map(lit => 
            typeof lit == 'string' ? lit : `${lit.get(0)}${lit.get(1)}`
        )];
        return `{${literalStrings.join(', ')}}`;
    })];
    return `{${clauseStrings.join(', ')}}`;
} 

In [ ]:
function test(s: string): string {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    return prettify(normalize(f));
}

In [ ]:
test('(¬p → q) → (p → q) → q');

In [ ]:
test('(a → b) ↔ (¬a ∧ ¬b)');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p');

In [ ]:
test('(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p');

In [ ]:
test('⊤');

In [ ]:
test('⊥');

In [ ]:
test("p → q");

In [ ]:
test("p ∧ (p → q)");

In [ ]:
test("(p ∧ q) → r");

In [ ]:
test("p ↔ q");

In [ ]:
test("(p → q) ∧ (q → r)");

In [ ]:
test("¬(p ∨ q)");

In [ ]:
test('p ∧ p');